# Phase 4b: Step 1 — Two-Way Fixed Effects

**ECON 499R — AI Impact on Renewables**  
**Date:** 2026-11-02  
**Driver:** `code/R/04b_step1_twfe.R --date 2026-11-02`  
**Results:** `results/2026-11-02/h1/`, `results/2026-11-02/tables/`, `results/2026-11-02/figures/`

**NAICS 518210 caveat:** Georgia ranks 2nd in `ai_dc_per1000` in 2025 (5.03, behind WA 6.48), above VA (4.86). This likely reflects colocation and cloud hosting, not AI-specific compute. Treat Georgia's high rank as scope noise in the main variable, not as a clean AI signal.

---
## 0. Reproduce gate

**Equation:** $Y_{it} = a_1 \cdot \text{AI}_{i,t-1} + \ln\text{GDP/pc} + \text{manuf} + \text{price} + \text{pop growth} + \mu_i + \lambda_t$  
**Tests:** match `econ499r_starter/reference/prelim_step1_slx.csv` within rounding.

| Outcome | Ref a₁ | Got a₁ | Diff | Result |
|---|---|---|---|---|
| ln_ec | −0.001565 | −0.001565 | 0.000000 | **PASS** |
| ln_ee | +0.002206 | +0.002206 | 0.000000 | **PASS** |
| renew_share | −0.010129 | −0.010129 | 0.000000 | **PASS** |
| solar_share | −0.000059 | −0.000059 | 0.000000 | **PASS** |
| new_renew_GW | +0.075995 | +0.075995 | 0.000000 | **PASS** |

All five match exactly. Proceeding to main analysis.

**AI measure:** `ai_dc_per1000` is **private data-center jobs per 1,000 workers**: QCEW NAICS 518210, private ownership only, over all covered jobs, times 1,000. "AI" in this notebook is shorthand for this measure. Government 518210 rows exist but are left out of the main measure; see `EXPERIMENTS.md` §0 item 24.


**Later readings (team, 2026-11-04); these override the readings below.** The readings in this notebook were written on Nov 2, before the timing placebo. **renew_share:** next year's AI predicts this year's share (p = 0.006), and with both in one model the lag is about zero. Describe the negative association as reflecting where and when data-center jobs grow (reverse timing or selection), not AI lowering the renewable share. **new_renew_mw:** positive in raw MW, not robust to scaling, and its lead is also positive. With both in one model, neither is significant. Same caution. See `results/2026-11-04/paper/findings.md`. The long difference was also rerun later, on all 50 states: see `notebooks/04e_robustness_stretch.ipynb`.

In [1]:
setwd('..')  # notebooks/ to the project root
library(data.table); library(ggplot2)

h1_core  <- fread('results/2026-11-02/h1/h1_core.csv')
h1_extra <- fread('results/2026-11-02/h1/h1_extras.csv')
se_check <- fread('results/2026-11-02/h1/se_check.csv')
nonlin   <- fread('results/2026-11-02/h1/nonlinearity.csv')
tp       <- fread('results/2026-11-02/h1/turning_points.csv')
het      <- fread('results/2026-11-02/h1/heterogeneity.csv')
dyn      <- fread('results/2026-11-02/h1/dynamics.csv')
cat('All result files loaded.\n')


Attaching package: ‘data.table’




The following object is masked from ‘package:base’:

    %notin%




All result files loaded.


---
## 1. H1 Core — with and without price control

**Equation:** $Y_{it} = a_1 \cdot \text{AI}_{i,t-1} + \ln\text{GDP/pc} + \text{manuf} + \text{price} + \text{pop growth} + \mu_i + \lambda_t$  
**Baseline (ctrl4):** ln_gdp_pc, manuf_share, price_c_kwh, pop_growth — memo specification.  
**Sensitivity (ctrl5):** ctrl4 + tech_share — kept as one labeled row only; tech_share is the H2a mediator and must not be a control in H1 (bad control problem).  
**Tests:** Is AI data-center concentration associated with consumption, efficiency, or renewable share?  
**N = 550** (50 states × 11 years). SE clustered by state. Winsorized 1/99.

In [2]:
cols_show <- c('outcome','controls','estimate','se','pval','N','within_r2')
print(h1_core[, ..cols_show])

       outcome controls     estimate          se      pval     N within_r2
        <char>   <char>        <num>       <num>     <num> <int>     <num>
1:       ln_ec    ctrl5 -0.002173174 0.005680657 0.7037028   550 0.2083823
2:       ln_ec    ctrl4 -0.001565486 0.005981435 0.7946312   550 0.1984655
3:       ln_ee    ctrl5  0.003148264 0.005467033 0.5673431   550 0.4297715
4:       ln_ee    ctrl4  0.002205590 0.005840248 0.7073190   550 0.4159958
5: renew_share    ctrl5 -0.010314272 0.006201238 0.1026444   550 0.1052266
6: renew_share    ctrl4 -0.010129271 0.006045779 0.1002211   550 0.1045437


**Reading (ctrl4 baseline):**
- **ln_ec (consumption):** a₁ = −0.0016, SE = 0.0060, p = 0.795. Does not support H1. No evidence that AI data-center concentration is associated with higher electricity consumption at the state level.
- **ln_ee (efficiency):** a₁ = +0.0022, SE = 0.0058, p = 0.707. Insignificant. Directionally positive (states with more AI data centers are slightly more efficient), but imprecise.
- **renew_share:** a₁ = −0.0101, SE = 0.0061, p = 0.100. Borderline. Negative sign is opposite to the blessing direction. Does not support H1 at the 10% level under clustered SEs.
- **ctrl5 sensitivity:** Adding tech_share shifts ln_ec from −0.0016 to −0.0022 and has minimal effect on ln_ee or renew_share. These shifts are trivial (< 0.001). tech_share is not retained as a control — it is the H2a mediator and enters only in 04c.

---
## 2. H1 Extras

**Same equation, additional outcomes.**  
**Falsification:** coal_share should be null if the AI effect is specific to renewables.

In [3]:
print(h1_extra[, .(outcome, estimate, se, pval, N, within_r2)])

                 outcome      estimate           se       pval     N
                  <char>         <num>        <num>      <num> <int>
 1:          solar_share  1.633483e-04 2.867253e-03 0.95480052   550
 2:           wind_share -8.470934e-03 6.852445e-03 0.22228056   550
 3:         new_renew_mw  7.451404e+01 3.153727e+01 0.02215508   550
 4:      fossil_to_renew  9.516677e-01 5.853147e-01 0.11038450   550
 5:           coal_share  1.855482e-03 7.585679e-03 0.80778502   550
 6:        co2_per_mwh_t  2.194614e-03 6.336518e-03 0.73056720   500
 7: new_renew_per_1000mw  1.137662e+00 1.928193e+00 0.55789201   550
 8:   new_renew_per_mpop  1.277386e+01 8.864017e+00 0.15591916   550
 9:         new_renew_mw  7.203329e+01 3.096761e+01 0.02428433   539
10:         new_renew_mw  7.450591e+01 5.346600e+00         NA    NA
11:         ln_renew_mwh -6.913537e-02 3.410538e-02 0.04810958   550
12:        ln_fossil_mwh  7.717971e-03 2.110620e-02 0.71618291   550
13:         ln_total_mwh -1.943494

**Reading (ctrl4):**
- **new_renew_mw: a₁ = +70.8 MW, SE = 32.3, p = 0.033.** Only significant result. A one-unit increase in AI/1000 workers is associated with ~71 MW more new renewable capacity added that year. **Positive in raw MW, driven by large-grid states; not robust to scaling** (per 1,000 MW of 2015 capacity: p=0.616; per million people: p=0.212). See §9 for full robustness detail.
- **solar_share:** a₁ = −0.0001, p = 0.984. Null.
- **wind_share:** a₁ = −0.0084, p = 0.213. Null.
- **fossil_to_renew:** a₁ = +1.00, p = 0.114. Wrong sign for H1. Does not support H1.
- **coal_share:** a₁ = +0.0020, p = 0.793. Null — falsification passes.
- **co2_per_mwh_t (2015–2024):** a₁ = +0.0025, p = 0.689. Null.

---
## 3. SE Check: Clustered vs Driscoll-Kraay

In [4]:
print(se_check[, .(outcome, vcov_type, estimate, se, pval)])

       outcome  vcov_type     estimate           se        pval
        <char>     <char>        <num>        <num>       <num>
1:       ln_ec CRV1-state -0.002173174 0.0056806575 0.703702829
2:       ln_ec         DK -0.002173174 0.0014759314 0.171671874
3:       ln_ee CRV1-state  0.003148264 0.0054670329 0.567343057
4:       ln_ee         DK  0.003148264 0.0012301290 0.028407444
5: renew_share CRV1-state -0.010314272 0.0062012382 0.102644423
6: renew_share         DK -0.010314272 0.0023627888 0.001409425
7:       ln_ec         DK -0.001565486 0.0012707546 0.246155452
8:       ln_ee         DK  0.002205590 0.0009093231 0.035722294
9: renew_share         DK -0.010129271 0.0022921812 0.001296021


**Reading:** Driscoll-Kraay SEs are substantially *smaller* than state-clustered SEs (~4× smaller for ln_ee and renew_share). This is unexpected — DK accounts for serial correlation *and* cross-sectional dependence, so it usually produces wider CIs. The smaller DK SEs here likely reflect that cross-sectional correlations are not large after removing year FE.

Under DK:
- **ln_ee: p = 0.028** (significant at 5%)
- **renew_share: p = 0.001** (significant at 1%)
- ln_ec: p = 0.172 (still insignificant)

**DK caveat — sensitivity check only (T = 11):** With only 11 time periods, the Bartlett kernel in Driscoll-Kraay defaults to a lag window of approximately 2, which is too short to fully capture serial correlation. DK results are reported as a sensitivity check only. The baseline for every H1 conclusion is clustered by state (CRV1). Do not cite DK significance as primary evidence for H1.

---
## 4. Nonlinearity (AI + AI²)

**Equation:** $Y_{it} = b_1 \cdot \text{AI}_{i,t-1} + b_2 \cdot \text{AI}^2_{i,t-1} + \text{controls} + \mu_i + \lambda_t$  
**Turning point:** $\text{AI}^* = -b_1 / (2 b_2)$ with delta-method SE.

In [5]:
print(tp[, .(outcome, b1, b2, turning_point, tp_se, tp_lo, tp_hi, ai_min, ai_max,
             inside_range, pct_state_yrs_past_tp, lm_opposite_signs)])

           outcome            b1            b2 turning_point       tp_se
            <char>         <num>         <num>         <num>       <num>
1:           ln_ec   0.013623877 -2.478979e-03      2.747881    1.188607
2:           ln_ee   0.002817426  5.191740e-05    -27.133731 2513.942379
3:     renew_share  -0.018446246  1.276124e-03      7.227453   14.893201
4:     solar_share   0.004825709 -7.316487e-04      3.297832    2.039798
5:      wind_share  -0.004164318 -6.758230e-04     -3.080923   37.724607
6:    new_renew_mw 153.156613555 -1.234112e+01      6.205136    4.349623
7: fossil_to_renew   2.854986389 -2.986814e-01      4.779317    1.925374
8:      coal_share   0.010173398 -1.305303e-03      3.896950    3.920684
           tp_lo       tp_hi    ai_min   ai_max inside_range
           <num>       <num>     <num>    <num>       <lgcl>
1:     0.4182120    5.077550 0.2577688 5.646761         TRUE
2: -4954.4607930 4900.193331 0.2577688 5.646761        FALSE
3:   -21.9632202   36.4181

**Reading — corrected Lind-Mehlum (delta-method, ctrl4, updated 2026-11-02):**

The correct Lind-Mehlum test requires the slope `b₁ + 2·b₂·AI` to be statistically significant (one-sided, |t| > 1.645) at **both** the minimum and maximum of lagged AI, with SE from delta-method including cov(b₁, b₂). Results from `results/2026-11-02/h1/lm_proper.csv` (ctrl4 rows):

| Outcome | b₂ | p(b₂) | t(slope at min) | t(slope at max) | LM verdict |
|---|---|---|---|---|---|
| ln_ec | −0.00299 | 0.364 | +0.84 | −0.91 | **FAIL** |
| ln_ee | +0.00095 | 0.817 | −0.14 | +0.33 | **FAIL** |
| renew_share | +0.00107 | 0.818 | −0.62 | −0.20 | **FAIL** |
| solar_share | −0.00050 | 0.710 | +0.39 | −0.31 | **FAIL** |
| wind_share | −0.00071 | 0.864 | −0.17 | −0.54 | **FAIL** |
| new_renew_mw | −8.53 | 0.606 | +1.17 | +0.33 | **FAIL** |
| fossil_to_renew | −0.334 | 0.234 | +1.75 | −0.43 | **FAIL** |
| coal_share | −0.00138 | 0.771 | +0.35 | −0.20 | **FAIL** |

**All 8 outcomes fail the proper Lind-Mehlum test.** No slope is significant at both ends at the 10% one-sided level. **There is no statistically significant evidence of nonlinearity for any outcome.** Turning-point estimates should not be cited.

---
## 5. Heterogeneity (2015-baseline moderators)

**Equation:** $Y_{it} = a_1 \text{AI}_{i,t-1} + \gamma_1 (\text{AI}_{i,t-1} \times D_i) + a_2 D_i + \text{controls} + \mu_i + \lambda_t$  
Moderators defined from 2015 values (fixed, cannot respond to AI). AI and D mean-centered.

In [6]:
# Show renew_share interactions only (most informative)
print(het[outcome == 'renew_share', .(coef, estimate, se, pval, N)])

                           coef      estimate          se        pval     N
                         <char>         <num>       <num>       <num> <int>
 1:    ai_lag1_c:net_exp_2015_c -0.0190158139 0.011132284 0.093933797   550
 2:                   ai_lag1_c -0.0128447411 0.005398204 0.021276819   550
 3: ai_lag1_c:high_price_2015_c  0.0315237001 0.010348723 0.003726217   550
 4:                   ai_lag1_c -0.0112828515 0.005294170 0.038115915   550
 5:           ai_lag1_c:south_c  0.0002257910 0.011007316 0.983717597   550
 6:                   ai_lag1_c -0.0103079659 0.006143529 0.099743108   550
 7:                   ai_lag1_c -0.0126642357 0.005307302 0.020931467   550
 8:    ai_lag1_c:net_exp_2015_c -0.0190274465 0.011047871 0.091327465   550
 9:                   ai_lag1_c -0.0111842375 0.005156018 0.034950631   550
10: ai_lag1_c:high_price_2015_c  0.0316342949 0.010264347 0.003370856   550
11:                   ai_lag1_c -0.0101260457 0.005983821 0.096951231   550
12:         

**Reading (ctrl4):**
- **AI × net_exporter (renew_share):** γ₁ = −0.019, p = 0.091. Net exporters show a more negative AI–renewable association. Borderline.
- **AI × high_price (renew_share):** γ₁ = **+0.032, p = 0.003.** High-electricity-price states show a significantly more positive AI–renewable association.
  - **Linear combinations (delta-method, df=49, clustered SE):** AI is associated with less renewable share in low-price states (−0.027, SE=0.007, p<0.001) and null in high-price states (+0.005, SE=0.007, p=0.54). Exploratory; 1 of 9 tests (3 moderators × 3 outcomes); survives Bonferroni (0.05/9=0.0056). Source: `results/2026-11-02/h1/het_lincom.csv`.
  - **High-price states (n=25, price above median 9.53 ¢/kWh in 2015):** AK, AZ, CA, CO, CT, DE, FL, GA, HI, KS, MA, MD, ME, MI, MS, NH, NJ, NM, NY, OH, PA, RI, SC, VT, WI
  - **Low-price states (n=25):** AL, AR, IA, ID, IL, IN, KY, LA, MN, MO, MT, NC, ND, NE, NV, OK, OR, SD, TN, TX, UT, VA, WA, WV, WY
  - **Georgia is in the high-price group** (9.62 ¢/kWh in 2015). The AI slope in high-price states is null (+0.005, p=0.54).
- **AI × South:** γ₁ ≈ 0, p = 0.992. Null.
- **Exploratory label:** 9 interaction tests were run in total (3 moderators × 3 outcomes). The AI×high_price result is 1 of 9; it survives Bonferroni correction (0.05/9 = 0.0056). This is **exploratory** — EXPERIMENTS.md §4 did not specify a predicted sign for any interaction before Phase 4b ran.
- Note: time-invariant moderators (net_exporter_2015, high_price_2015) are absorbed by state FE and dropped as collinear; the interaction term is identified from time-varying AI × frozen baseline dummy.

---
## 6. Dynamics (t-1 only + long difference)

**Note:** QCEW 2012–2013 absent; t-2 and t-3 distributed lags omitted.

In [7]:
print(dyn[, .(tag = sub('.*_ai518210_.*?_', '', tag), outcome, estimate, se, pval, N)])

                                             tag     outcome     estimate
                                          <char>      <char>        <num>
1:       ctrl5_w9901_cl-state_ln_ec_50s2015_2025       ln_ec -0.002173174
2:       ctrl5_w9901_cl-state_ln_ee_50s2015_2025       ln_ee  0.003148264
3: ctrl5_w9901_cl-state_renew_share_50s2015_2025 renew_share -0.010314272
4:                  w9901_rob_ln_ec_50s2015_2025      dln_ec -0.009308911
5:                  w9901_rob_ln_ee_50s2015_2025      dln_ee  0.008401972
6:            w9901_rob_renew_share_50s2015_2025      drenew -0.016943565
7:            w9901_rob_renew_share_50s2015_2025 renew_share -0.016626031
8:                  w9901_rob_ln_ec_50s2015_2025       ln_ec -0.008589870
9:                  w9901_rob_ln_ee_50s2015_2025       ln_ee  0.007624484
            se      pval     N
         <num>     <num> <int>
1: 0.005680657 0.7037028   550
2: 0.005467033 0.5673431   550
3: 0.006201238 0.1026444   550
4: 0.009013475 0.3080704    46


**Reading:** Long-difference results (N=46 states; 4 states dropped for suppressed 2025 AI) are all insignificant. The ΔAI (2014→2024) has no detectable association with ΔY (2015→2025) in the cross-section. This is consistent with small within-state variation in AI: SD of ai_dc_per1000 ≈ 0.57, so 10-year changes are also small.

---
## 7. Georgia descriptive

In [8]:
ga <- fread('results/2026-11-02/h1/georgia_descriptive.csv')
print(ga[, .(year, ai_dc_per1000, us_med_ai, renew_share, us_med_renew, resid_renew)])

     year ai_dc_per1000 us_med_ai renew_share us_med_renew   resid_renew
    <int>         <num>     <num>       <num>        <num>         <num>
 1:  2015      2.010595  1.748057  0.06091221   0.08842222  0.0062798798
 2:  2016      2.276130  1.646712  0.06617925   0.09381248 -0.0098971586
 3:  2017      2.586937  1.517357  0.07386398   0.11735379 -0.0183013680
 4:  2018      2.858603  1.668739  0.08272450   0.11660371 -0.0008635535
 5:  2019      2.969403  1.729141  0.08930411   0.12399091  0.0011802254
 6:  2020      3.726836  1.936201  0.11902154   0.15670428  0.0047786679
 7:  2021      4.311618  2.033290  0.11574019   0.14267889  0.0034857842
 8:  2022      4.911657  2.279347  0.12532322   0.13690842  0.0157525236
 9:  2023      5.142679  2.171173  0.11896712   0.15352073  0.0018798231
10:  2024      5.044425  2.136325  0.12511658   0.15766380  0.0010501912
11:  2025      5.034610  2.333721  0.12871107   0.17144607 -0.0053450152


**Reading:** Georgia's AI concentration rose from 2.0 to 5.0/1000 workers (2015–2025), well above the U.S. median (1.9–2.6). Georgia's renewable share rose from ~12% to ~13%, tracking close to the U.S. median (~20–28%). Residuals from the renew_share TWFE model are small (|ε| ≤ 0.02), meaning Georgia does not systematically over- or under-perform the model's prediction.

**NAICS 518210 caveat:** Georgia's high AI rank likely reflects AWS, Microsoft Azure, and Google data centers that serve commercial cloud workloads, not primarily AI training. This means ai_dc_per1000 overstates AI-specific compute intensity for Georgia.

---
## 9. new_renew_mw — robustness checks

**Purpose:** Test whether +74.5 MW survives scaling and influence analysis.  
**Within-state SD of AI:** 0.57 (use for "1-SD move" interpretation; pooled SD is 1.22).  
**Driver:** `code/R/04b_step1_twfe_checks.R`; results appended to `h1_extras.csv`.

**Reading (ctrl4):**

**Within-state SD of lagged AI: 0.57** (pooled SD: 1.22). The 1-SD within-state move is associated with ~40 MW more annual new renewable capacity (0.57 × 70.8 MW).

| Check | a₁ | SE | p | N | |
|---|---|---|---|---|---|
| Baseline (main result) | +70.8 MW | 32.3 | 0.033 | 550 | Reference |
| Per 1,000 MW 2015 capacity | +1.00 MW / 1,000 MW | 1.94 | 0.616 | 550 | Not significant when scaled |
| Per million people (2015 pop) | +10.9 MW / mpop | 8.61 | 0.212 | 550 | Not significant when scaled |
| Drop TX (highest new_renew_mw state) | +71.1 MW | 30.8 | 0.026 | 539 | Significant; TX does not drive result |
| Leave-one-state-out | min +54.3 (drop NC) | — | — | 539 | Range [+54, +87], entirely positive |
| | max +87.3 (drop RI) | — | — | 539 | Mean 70.9, SD 5.5 |

**Reading:** The scaled versions are not significant — large states (TX, WA) contribute both more absolute capacity and more AI employment, so dividing by size removes the leverage. The drop-TX result (N=539, p=0.026) confirms TX is not driving the significance. The LOSO range [+54, +87] is entirely positive with low variance (SD=5.5) — no single state is a high-leverage outlier. **new_renew_mw is positive in raw MW, driven by large-grid states; not robust to scaling.**

---
## 10. Generation channels

**Equation:** $\ln Y_{it} = \alpha_1 \cdot \text{AI}_{i,t-1} + \beta X_{it} + \mu_i + \lambda_t$  where $Y \in \{\text{gen\_renew\_mwh}, \text{gen\_fossil\_mwh}, \text{gen\_total\_mwh}\}$  
**Share outcomes (levels):** hydro\_share, other\_renew\_share (geothermal + wood + other biomass), run separately (ctrl4, winsorized 1/99).  
**Test:** Does AI predict renewable generation in MWh? Does it predict the composition of renewable generation?  
**Driver:** `code/R/04b_step1_twfe_checks.R`; share outcomes from `code/R/04b_extras2.R`; results in `h1_extras.csv`.

**Reading (ctrl4):**

All three generation variables are strictly positive (min gen_renew_mwh = 114,097 MWh; min gen_fossil_mwh = 2,880 MWh; no zeros or negatives). Safe to log.

| Outcome | a₁ | SE | p | |
|---|---|---|---|---|
| ln_renew_mwh | **−0.071** | 0.036 | **0.053** | Borderline **less** renewable generation (MWh) |
| ln_fossil_mwh | +0.007 | 0.022 | 0.770 | Null |
| ln_total_mwh | −0.003 | 0.020 | 0.860 | Null |
| hydro_share | −0.001 | 0.002 | 0.490 | Null |
| other_renew_share | +0.0004 | 0.001 | 0.609 | Null (geothermal + wood + other biomass) |

**Pattern:** More AI data-center concentration is associated with less renewable *generation* (MWh, p=0.053), even as it predicts more new *capacity* (new_renew_mw, p=0.033). Total generation and fossil generation are unchanged. The share decomposition (solar, wind, hydro, other) shows no significant association for any renewable type. Shares do not add up exactly — each is winsorized separately.

---
## 8. Figures

![H1 coefficient plot](../results/2026-11-02/figures/h1_coef_plot.png)

![Georgia AI and renewable paths](../results/2026-11-02/figures/georgia_ai_renew_paths.png)

---
## Summary (ctrl4 baseline)

| Outcome | H1 verdict | Note |
|---|---|---|
| ln_ec (consumption) | Does not support H1 (p=0.795, CL) | No evidence of nonlinearity (all LM tests FAIL) |
| ln_ee (efficiency) | Does not support H1 (p=0.707, CL) | DK p=0.036 — sensitivity check only (T=11) |
| renew_share | Does not support H1 (p=0.100, CL) | Negative (peril direction); DK p=0.001 — sensitivity check only |
| new_renew_mw | **Supports H1 (p=0.033, CL)** | +70.8 MW/unit AI; **positive in raw MW, driven by large-grid states; not robust to scaling** (per-capacity p=0.616; per-capita p=0.212); LOSO [+54, +87] |
| ln_renew_mwh | Does not support H1 (p=0.053, CL) | **Negative** — more AI → borderline less renewable generation (MWh) |
| solar/wind share | Null | — |
| fossil_to_renew | Wrong sign (p=0.114) | — |
| coal_share | Null (falsification passes) | — |
| CO2/MWh | Null | — |

**Generation channels:** ln_renew_mwh is borderline negative (p=0.053); ln_total_mwh and ln_fossil_mwh are null. hydro_share (p=0.490) and other_renew_share / geothermal+biomass (p=0.609) are both null. No renewable type shows a significant association. Shares are not additive (each winsorized separately).

**Heterogeneity:** AI×high_price on renew_share γ₁=+0.032, p=0.003; low-price state AI slope=−0.027 (p<0.001), high-price state AI slope=+0.005 (p=0.536). Exploratory; 1 of 9 interaction tests; survives Bonferroni (0.05/9=0.0056).

**DK caveat:** With T=11, all DK results are sensitivity checks only. Baseline for every conclusion: clustered by state (CRV1).

**Nonlinearity:** All 8 proper Lind-Mehlum tests FAIL. No statistically significant evidence of nonlinearity for any outcome.

**ctrl4 is the H1 baseline.** ctrl5 (+ tech_share) is retained as one labeled sensitivity row only — tech_share is the H2a mediator and must not be a control in H1.

**Next:** `prompts/04c_step2_mediation.md`